# Task 1: Web Scraping
### Evelyn Valeria Sarmiento Vásquez

In [ ]:
#Instalando todas las librerías necesarias: 
#!python -m pip install selenium webdriver-manager lxml unidecode


   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.6 MB 6.3 MB/s eta 0:00:02
   ------------ --------------------------- 2.9/9.6 MB 8.8 MB/s eta 0:00:01
   -------------------- ------------------- 5.0/9.6 MB 9.6 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.6 MB 9.6 MB/s eta 0:00:01
   ------------------------------------ --- 8.7/9.6 MB 9.3 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 8.9 MB/s  0:00:01

   ---------------------------------------- 0/9 [wsproto]
  Attempting uninstall: urllib3
   ---------------------------------------- 0/9 [wsproto]
    Found existing installation: urllib3 2.5.0
   ---------------------------------------- 0/9 [wsproto]
    Uninstalling urllib3-2.5.0:
   ---------------------------------------- 0/9 [wsproto]
      Successfully uninstalled urllib3-2.5.0
   ---------------------------------------- 0/9 [wsproto]
   ---- -----------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
# Importando todas las librerías necesarias
import base64
import time
import os

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

In [7]:
#Abriendo la url de la UNMSM:

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

driver.maximize_window()

url='https://admision.unmsm.edu.pe/Website20262/A/A.html'

driver.get(url)


## Funciones auxiliares

In [ ]:
# ── Función 1: decodificar nombres ofuscados ──────────────────────────────
# El sitio de UNMSM guarda los nombres en base64 dentro del atributo data-auth
# para evitar scrapers simples.  Esta función los decodifica a texto legible.

def decode_obfuscated(data_auth):
    try:
        return base64.b64decode(data_auth).decode("utf-8")
    except Exception:
        return data_auth  # Si falla, devuelve el valor original


# ── Función 2: mostrar TODAS las filas de golpe ───────────────────────────
# DataTables carga todos los datos en memoria pero solo muestra 50 por página.
# Con una línea de JavaScript le pedimos que quite el límite de paginación
# y dibuje todas las filas de una vez → no necesitamos hacer clic en "Siguiente".

def show_all_rows(driver):
    driver.execute_script(
        "try { $('#tablaPostulantes').DataTable().page.len(-1).draw(); }"
        " catch(e) {}"
    )
    time.sleep(2)  # Esperamos que DataTables re-renderice todas las filas


# ── Función 3: extraer filas de la tabla ──────────────────────────────────
# Recorre cada <tr> del tbody, decodifica las celdas ofuscadas y devuelve
# una lista de listas con los datos limpios.

def extract_all_rows(driver):
    show_all_rows(driver)
    rows = []
    tbody = driver.find_element(By.CSS_SELECTOR, "#tablaPostulantes tbody")
    for tr in tbody.find_elements(By.TAG_NAME, "tr"):
        tds = tr.find_elements(By.TAG_NAME, "td")
        row = []
        for td in tds:
            obf = td.find_elements(By.CLASS_NAME, "obfuscated")
            if obf:
                row.append(decode_obfuscated(obf[0].get_attribute("data-auth")))
            else:
                row.append(td.text.strip())
        if row:
            rows.append(row)
    return rows


# ── Función 4: scrapear una carrera completa ──────────────────────────────
# Abre la página de resultados de una carrera, espera que la tabla cargue,
# expande todas las filas y las devuelve con el nombre de la carrera al inicio.

def scrape_career(driver, url, career_name):
    driver.get(url)
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.ID, "tablaPostulantes"))
    )
    time.sleep(1)  # Pequeña pausa para que DataTables termine de inicializar
    rows = extract_all_rows(driver)
    return [[career_name] + row for row in rows]

## Obtener los enlaces de todas las carreras

In [ ]:
# La página índice lista todas las carreras con un link a su tabla de resultados.
# Extraemos el nombre y la URL de cada una.

INDEX_URL = "https://admision.unmsm.edu.pe/Website20262/A/A.html"

driver.get(INDEX_URL)
WebDriverWait(driver, 15).until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a[href*='results.html']"))
)

links = driver.find_elements(By.CSS_SELECTOR, "a[href*='results.html']")
careers = [
    (link.text.strip(), link.get_attribute("href"))
    for link in links
    if link.text.strip()
]

print(f"Total de carreras encontradas: {len(careers)}")
for name, url in careers[:5]:          # Muestra las primeras 5 como verificación
    print(f"  {name}  →  {url}")

## Scrapear todas las carreras y consolidar los datos

In [ ]:
# Recorre cada carrera, extrae TODOS sus postulantes (sin límite de paginación)
# y los acumula en all_data.

COLUMNS = ["Carrera", "Código", "Apellidos y Nombres", "Escuela", "Puntaje", "Mérito E.P", "Observación"]

all_data = []

for i, (career_name, url) in enumerate(careers, start=1):
    print(f"[{i:>3}/{len(careers)}] {career_name} ...", end=" ", flush=True)
    try:
        rows = scrape_career(driver, url, career_name)
        all_data.extend(rows)
        print(f"{len(rows)} postulantes")
    except Exception as e:
        print(f"ERROR: {e}")

print(f"\nTotal de filas recolectadas: {len(all_data)}")

## Guardar en Excel

In [ ]:
# Construye el DataFrame y lo exporta a Excel dentro de la carpeta output/

os.makedirs("output", exist_ok=True)
OUTPUT_PATH = os.path.join("output", "resultados_sanmarcos.xlsx")

df = pd.DataFrame(all_data, columns=COLUMNS)
df.to_excel(OUTPUT_PATH, index=False)

print(f"Archivo guardado: {OUTPUT_PATH}")
print(f"Total filas: {len(df)}")
df.head()  # Muestra las primeras filas como verificación

In [ ]:
# Cerrar el navegador al terminar
driver.quit()